In [1]:
from PIL import Image, ImageOps
from pathlib import Path
import torch
from torch import nn
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
checkpoint = torch.load(
    "checkpoints/resnet.pth",
    map_location="cpu",
    weights_only=True,
)

class_names = checkpoint["class_names"]

model = resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    len(class_names),
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

<All keys matched successfully>

In [3]:
weights = ResNet18_Weights.IMAGENET1K_V1
resnet_transform = weights.transforms()

In [4]:

@torch.inference_mode()
def predict_image(
    image_path,
    model,
    transform,
    class_names,
    top_k = 3
):

  model.eval()
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)

  with Image.open(image_path) as source:
    image = ImageOps.exif_transpose(source).convert('RGB')

  inputs = transform(image).unsqueeze(0).to(device)

  logits = model(inputs)
  probabilities = torch.softmax(logits, dim=1)

  top_probabilities, top_indices = probabilities.topk(
      top_k,
      dim=1
  )

  return [
      {
          "class_names": class_names[index],
          "probability": probability
      }
      for index, probability in zip(
          top_indices[0].cpu().tolist(),
          top_probabilities[0].cpu().tolist()
      )
  ]

In [5]:
custom_image_1 = {
    'label': "Japanese_Chin",
    "path": Path("/content/Japanese_Chin.jpg")
}
custom_image_2 = {
    'label': "Yorkshire_Terrier",
    "path": Path("/content/Yorkshire_Terrier.jpg")
}
custom_image_3 = {
    'label': "Keeshond",
    "path": Path("/content/Keeshond.jpg")
}
custom_image_4 = {
    'label': "egyptian",
    "path": Path("//content/egyptian.jpg")
}
custom_image_5 = {
    'label': "persian",
    "path": Path("/content/persian.jpg")
}


images = [
    custom_image_1,
    custom_image_2,
    custom_image_3,
    custom_image_4,
    custom_image_5
]

for image in images:
  predictions = predict_image(
      image_path=image['path'],
      model=model,
      transform=resnet_transform,
      class_names=class_names
  )

  for prediction in predictions:
    print(
        f"Real: {image["label"]} \n"
        "Predicted: \n"
        f"{prediction["class_names"]}: "
        f"{prediction["probability"]:.2%}"
    )

  print()

Real: Japanese_Chin 
Predicted: 
Japanese Chin: 97.61%
Real: Japanese_Chin 
Predicted: 
Havanese: 1.07%
Real: Japanese_Chin 
Predicted: 
Ragdoll: 0.69%

Real: Yorkshire_Terrier 
Predicted: 
Yorkshire Terrier: 100.00%
Real: Yorkshire_Terrier 
Predicted: 
Pomeranian: 0.00%
Real: Yorkshire_Terrier 
Predicted: 
Havanese: 0.00%

Real: Keeshond 
Predicted: 
Keeshond: 90.14%
Real: Keeshond 
Predicted: 
Pomeranian: 7.43%
Real: Keeshond 
Predicted: 
Birman: 1.44%

Real: egyptian 
Predicted: 
Egyptian Mau: 99.48%
Real: egyptian 
Predicted: 
Bengal: 0.50%
Real: egyptian 
Predicted: 
Maine Coon: 0.01%

Real: persian 
Predicted: 
Persian: 81.44%
Real: persian 
Predicted: 
Ragdoll: 15.50%
Real: persian 
Predicted: 
Birman: 2.29%

